# 🚀 Hướng dẫn Host LLM API (Qwen-2.5) trên Google Colab

Notebook này giúp bạn biến Google Colab (với GPU T4 miễn phí) thành một **Inference Server** cục bộ. Nó sẽ cung cấp một API tương thích 100% với OpenAI để Backend của bạn gọi tới mà không lo bị Rate Limit.

**Yêu cầu trước khi bắt đầu:**
1. Vào menu `Runtime` > `Change runtime type` > Chọn Hardware accelerator là **T4 GPU**.
2. Đăng ký một tài khoản miễn phí tại [ngrok.com](https://ngrok.com) và lấy Authtoken.

### Bước 1: Cài đặt Ollama
Ollama là công cụ giúp tải và chạy LLM cực kỳ dễ dàng, tối ưu tốt cho bộ nhớ GPU.
(Lưu ý: Môi trường Colab hiện tại cần cài thêm gói `zstd` để giải nén Ollama)

In [ ]:
!sudo apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

### Bước 2: Khởi động Ollama Server chạy ngầm
Mặc định Ollama sẽ chạy ở cổng `11434`. Ta cần cấu hình CORS và Host để nó cho phép Ngrok truy cập.

In [ ]:
!pkill -9 ollama
import subprocess
import time
import os

# Cấu hình bảo mật để Ngrok có thể gọi được vào Ollama
env = os.environ.copy()
env['OLLAMA_ORIGINS'] = '*'
env['OLLAMA_HOST'] = '0.0.0.0'

# Mở Ollama dưới dạng background process
process = subprocess.Popen(['ollama', 'serve'], env=env)
time.sleep(3) # Đợi 3 giây để server khởi động hoàn tất
print("✅ Ollama server đã khởi động ngầm!")

### Bước 3: Tải mô hình Qwen-2.5-7B
Alibaba Qwen 2.5 là model hiểu Tiếng Việt và làm toán/logic cực tốt. Quá trình tải sẽ mất khoảng 1-2 phút (dung lượng ~4.7GB).

In [ ]:
!ollama pull qwen2.5:7b

### Bước 4: Mở API ra thế giới bằng Ngrok
Localhost của Colab không thể truy cập từ máy tính của bạn. Ta dùng `ngrok` để tạo một đường hầm (tunnel).

**Chú ý:** Dán Ngrok Authtoken của bạn vào biến `NGROK_TOKEN` bên dưới.

In [ ]:
!pip install pyngrok -q
from pyngrok import ngrok

# TẠO TÀI KHOẢN NGROK VÀ PASTE TOKEN VÀO ĐÂY:
NGROK_TOKEN = "YOUR_NGROK_AUTH_TOKEN_HERE"

ngrok.set_auth_token(NGROK_TOKEN)

# Tạo tunnel tới cổng 11434 của Ollama với host_header=localhost
public_url = ngrok.connect(11434, bind_tls=True, host_header="localhost:11434").public_url
print("\n" + "="*50)
print(f"🔥 ĐÂY LÀ ĐỊA CHỈ API CỦA BẠN:")
print(f"👉 {public_url}/v1")
print("="*50 + "\n")
print("HÃY COPY LINK NÀY VÀ ĐƯA VÀO FILE .env Ở BACKEND CỦA BẠN.")

### 🎉 Xong! Hệ thống đã sẵn sàng.
Bây giờ bạn cứ để nguyên tab Colab này mở (có thể thỉnh thoảng click vào để nó không bị sập ngủ đông).

Ở Backend máy tính của bạn, dùng code Python sau để test thử:

In [ ]:
# Đừng chạy ô này trên Colab, hãy chạy ở máy tính local của bạn để test kết nối!
"""
import requests
import json

url = "ĐIỀN_LINK_NGROK_CỦA_BẠN/v1/chat/completions"
headers = {
    "Content-Type": "application/json",
    "ngrok-skip-browser-warning": "true"
}

payload = {
    "model": "qwen2.5:7b",
    "messages": [
        {"role": "system", "content": "Bạn là AI phân tích thực thể Tiên hiệp. Chỉ xuất JSON."},
        {"role": "user", "content": "Phân tích câu: Lâm Nguyệt Nhi rút Băng Lam Kiếm ra chém một đòn Huyết Ma Trảm."}
    ]
}

response = requests.post(url, headers=headers, json=payload)
print(response.json())
"""